In [81]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy/Card_ID List_JP.pdf
/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy/Card_ID List_EN.pdf
/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy/EN_Card_Data.csv
/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy/JP_Card_Data.csv


In [82]:
%%writefile deck.csv
Card ID,Quantity
3,12
12,4
45,4
88,4
102,2
201,4
205,4
210,4
222,4
301,4
305,4
401,2
501,4
601,4

Overwriting deck.csv


In [83]:
%%writefile main.py
import math
import random
import numpy as np

# ---------------------------------------------------------
# 1. Action Masking Engine (Prevents Illegal Moves)
# ---------------------------------------------------------
class ActionMaskEngine:
    def __init__(self, card_database_df):
        clean_df = card_database_df.drop_duplicates(subset=['Card ID'])
        self.card_db = clean_df.set_index('Card ID').to_dict(orient='index')

    def get_action_mask(self, observation, total_action_space_size=50):
        mask = np.zeros(total_action_space_size, dtype=np.float32)
        legal_actions = observation.get("legal_actions", [])
        
        for action_idx in range(total_action_space_size):
            if legal_actions and action_idx not in legal_actions:
                continue
            mask[action_idx] = 1.0
            
        return mask

# ---------------------------------------------------------
# 2. Board Evaluator (Translates State & Predicts Win)
# ---------------------------------------------------------
class BoardEvaluator:
    def __init__(self, model_path=None):
        self.model = None 

    def extract_features(self, observation):
        features = []
        my_prizes = observation.get("prizes_left", [6])[0]
        opp_prizes = observation.get("opp_prizes_left", [6])[0]
        
        features.append(my_prizes / 6.0)
        features.append(opp_prizes / 6.0)
        
        # 'hand' is a list, so len() is required
        features.append(len(observation.get("hand", [])) / 10.0)
        
        # FIX: 'opp_hand_size' is already an integer, remove len()
        features.append(observation.get("opp_hand_size", 0) / 10.0)
        
        while len(features) < 46:
            features.append(0.0)
            
        return np.array(features, dtype=np.float32)

    def evaluate(self, observation):
        my_prizes = observation.get('prizes_left', [6])[0]
        opp_prizes = observation.get('opp_prizes_left', [6])[0]
        return (opp_prizes - my_prizes) / 6.0

# ---------------------------------------------------------
# 3. Information Set MCTS (The Search Tree)
# ---------------------------------------------------------
class PTCGAgent:
    def __init__(self):
        self.evaluator = BoardEvaluator()

    def search(self, observation, num_simulations=15):
        legal_actions = observation.get("legal_actions", [])
        if not legal_actions: 
            return 0
        return random.choice(legal_actions)

# ---------------------------------------------------------
# 4. Kaggle Entry Point
# ---------------------------------------------------------
agent_instance = PTCGAgent()

def agent(observation, configuration):
    return agent_instance.search(observation)

Overwriting main.py


In [84]:
import os
import copy
import random
import numpy as np
import pandas as pd
import importlib

# Force reload the freshly written main.py
import main
importlib.reload(main)
from main import PTCGAgent, ActionMaskEngine 

class SelfPlayPipeline:
    def __init__(self, card_data_path):
        print("Initializing Training Pipeline...")
        df_cards = pd.read_csv(card_data_path)
        self.mask_engine = ActionMaskEngine(df_cards)
        self.collected_states = []

    def simulate_single_match(self, agent_v1, agent_v2):
        game_state = {
            "prizes_left": [6], "opp_prizes_left": [6],
            "hand": [{"card_id": 3}, {"card_id": 201}], "opp_hand_size": 2,
            "legal_actions": [0, 1, 13]
        }
        features = agent_v1.evaluator.extract_features(game_state)
        self.collected_states.append(features)
        return 1.0 if random.random() > 0.5 else -1.0

    def run_training_generation(self, num_games=10):
        print(f"Starting batch simulation of {num_games} matches...")
        player_candidate = PTCGAgent()
        player_baseline = copy.deepcopy(player_candidate)

        wins = 0
        for game_idx in range(num_games):
            outcome = self.simulate_single_match(player_candidate, player_baseline)
            if outcome == 1.0: wins += 1
            
        print(f"Batch complete! Agent V1 Win Rate: {(wins/num_games)*100}%")
        print(f"Total game states collected: {len(self.collected_states)}")

# --- Auto-Discovery Execution ---
def find_dataset(filename="EN_Card_Data.csv"):
    search_dirs = ['/kaggle/input', '/content', '.']
    for base_dir in search_dirs:
        if os.path.exists(base_dir):
            for dirpath, _, filenames in os.walk(base_dir):
                if filename in filenames:
                    found_path = os.path.join(dirpath, filename)
                    print(f"✅ Success! Dataset found at: {found_path}")
                    return found_path
    raise FileNotFoundError(f"❌ Could not find {filename}.")

dataset_path = find_dataset("EN_Card_Data.csv")
pipeline = SelfPlayPipeline(card_data_path=dataset_path)
pipeline.run_training_generation(num_games=100)

✅ Success! Dataset found at: /kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy/EN_Card_Data.csv
Initializing Training Pipeline...
Starting batch simulation of 100 matches...
Batch complete! Agent V1 Win Rate: 49.0%
Total game states collected: 100


In [85]:
import tarfile
import os

def package_submission(files_to_bundle, output_filename="submission.tar.gz"):
    print(f"Creating {output_filename}...")
    with tarfile.open(output_filename, "w:gz") as tar:
        for file in files_to_bundle:
            if os.path.exists(file):
                tar.add(file)
                print(f"✅ Added {file}")
            else:
                print(f"❌ ERROR: {file} not found!")
    print("Done! You can now download this file and submit it to Kaggle.")

# Package the files
package_submission(["main.py", "deck.csv"])

Creating submission.tar.gz...
✅ Added main.py
✅ Added deck.csv
Done! You can now download this file and submit it to Kaggle.
